In [0]:
# ! pip install vermin

In [0]:
import dataiku
import tempfile
import os
import logging
from typing import Dict, List, Tuple, Any

# Vermin imports for static analysis
try:
    from vermin import detect_paths, detect, Config
except ImportError:
    raise ImportError("The 'vermin' package is not installed. Please add 'vermin' to your requirements.")

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger()

class RecipeAnalyzer:
    """
    A class to analyze Dataiku Python recipes for Python version compatibility using Vermin.
    """

    def __init__(self):
        self.client = dataiku.api_client()
        self.project = self.client.get_project("DEPRECATED_TESTS_DSS_V12")
        self.vermin_config = Config()
        self.vermin_config.set_verbose(2)  # Suppress internal vermin logging

    def get_python_recipes(self) -> List[Dict[str, Any]]:
        """
        Retrieves a list of all Python code recipes in the current project.
        """
        recipes = self.project.list_recipes()
        # Filter strictly for 'python' type recipes
        return [r for r in recipes if r['type'] == 'python']

    def get_recipe_code(self, recipe_name: str) -> str:
        """
        Fetches the actual Python script content from a specific recipe.
        """
        recipe = self.project.get_recipe(recipe_name)
        settings = recipe.get_settings()
        
        # In Dataiku API, the script content is usually accessible via get_code() 
        # on the settings object for code recipes.
        try:
            return settings.get_code()
        except AttributeError:
            logger.warning(f"Could not retrieve code for {recipe_name}. It may not be a standard code recipe.")
            return ""

    def analyze_code_compatibility(self, code_content: str) -> Dict[str, str]:
        """
        Writes code to a temp file and runs Vermin analysis to find min/incompatible versions.
        """
        logger.info(f"Start analyze_code_compatibility")
        if not code_content.strip():
            logger.warning(f"No code detected")            
            return {"min_versions": "N/A", "incompatible_versions": "N/A"}

        # Vermin analyzes files, so we write the recipe code to a temporary file
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as tmp_file:
            # logger.info(f"Writing temp file")
            tmp_file.write(code_content)
            tmp_path = tmp_file.name

        try:
            # detect_paths returns: (mins, incomp, unique_versions, backports)
            # mins is a list of tuples, e.g. [(2, 7), (3, 6)]
            vermin_results = detect(code_content, config=self.vermin_config)            
            mins = vermin_results[0]
            incomp = vermin_results[1]
            # Format Minimum Versions (e.g., "2.7, 3.6")
            print(f"vermin_results: {vermin_results} --> mins: {mins}")
            # Example output from line above:
            # vermin_results: [(0, 0), (0, 0)] --> mins: (0, 0)
            # info: `print "Checking system configuration..."` requires 2.0 requires 2.0, !3
            # vermin_results: [(2, 0), None] --> mins: (2, 0)
            # vermin_results: [None, (3, 6)] --> mins: None
            # vermin_results: [None, (3, 6)] --> mins: None
            # vermin_results: [(0, 0), (0, 0)] --> mins: (0, 0)            
            
            # next line causes an exception: 'int' object is not subscriptable
            # Error analyzing code: 

            min_ver_str = ", ".join([f"{v[0]}.{v[1]}" if v[1] is not None else f"{v[0]}.0" for v in mins])
            logger.info(f"Analyze results 3...")            
            # Format Incompatible Versions
            incomp_ver_str = ", ".join([f"{v[0]}.{v[1] if len(v) > 1 else 'x'}" for v in incomp])
            logger.info(f"Analyze results 4...")
            if not incomp_ver_str:
                logger.warning(f"Incompat A")
                incomp_ver_str = "None"

            logger.info(f"End of analyze_code_compatibility")
            return {
                "min_versions": min_ver_str,
                "incompatible_versions": incomp_ver_str
            }

        except Exception as e:
            logger.error(f"Error analyzing code: {e}")
            return {"min_versions": "Error", "incompatible_versions": "Error"}
        
        finally:
            # Cleanup temp file
            if os.path.exists(tmp_path):
                os.remove(tmp_path)

    def run(self):
        """
        Main execution method to list and analyze recipes.
        """
        logger.info(f"Scanning project key: {self.project.project_key} for Python recipes...")
        python_recipes = self.get_python_recipes()
        
        if not python_recipes:
            logger.info("No Python recipes found in this project.")
            return

        logger.info(f"Found {len(python_recipes)} Python recipes. Beginning analysis...\n")
        
        # Table Header
        header = f"{'Recipe Name':<40} | {'Min Required':<20} | {'Incompatible':<20}"
        logger.info(header)
        logger.info("-" * len(header))

        for recipe_meta in python_recipes:
            name = recipe_meta['name']
            code = self.get_recipe_code(name)
            
            if code:
                result = self.analyze_code_compatibility(code)
                logger.info(f"{name:<40} | {result['min_versions']:<20} | {result['incompatible_versions']:<20}")
            else:
                logger.info(f"{name:<40} | {'Skipped (No Code)':<20} | {'-'}")

# --- Execution Entry Point ---
if __name__ == "__main__":
    analyzer = RecipeAnalyzer()
    analyzer.run()